In [0]:
import requests
import pandas as pd

# ---------------------------------------------------------
# Load the latest Eurostat snapshot from GitHub
# ---------------------------------------------------------

url = (
    "https://raw.githubusercontent.com/"
    "cicciosar/financial-data-platform/"
    "main/data/raw/nasa_10_f_bs.json"
)

response = requests.get(url, timeout=30)
response.raise_for_status()

data = response.json()

print("Dataset:", data["label"])
print("Source:", data["source"])
print("Updated:", data["updated"])
print("Dimensions:", data["id"])
print("Sizes:", data["size"])

In [0]:
# ---------------------------------------------------------
# Convert Eurostat JSON-stat 2.0 into observed records only
# ---------------------------------------------------------

dimensions = data["id"]
sizes = data["size"]
dimension_metadata = data["dimension"]

values = data.get("value", {})
status_values = data.get("status", {})


def ordered_categories(dimension_code):
    """
    Return category codes in the exact JSON-stat order.
    """
    category = dimension_metadata[dimension_code]["category"]
    index = category.get("index")

    if isinstance(index, dict):
        return [
            code
            for code, position in sorted(
                index.items(),
                key=lambda x: x[1]
            )
        ]

    if isinstance(index, list):
        return index

    labels = category.get("label", {})
    return list(labels.keys())


def to_indexed_map(container):
    """
    Convert JSON-stat value/status data into:
        flat_index -> observation
    Handles both sparse dictionaries and lists.
    """

    if isinstance(container, dict):
        return {
            int(key): value
            for key, value in container.items()
        }

    if isinstance(container, list):
        return {
            i: value
            for i, value in enumerate(container)
            if value is not None
        }

    return {}


def decode_flat_index(flat_index, sizes):
    """
    Convert a JSON-stat flat index into one position
    for each dimension.

    The last dimension is the fastest-changing dimension.
    """

    positions = [0] * len(sizes)
    remaining = flat_index

    for i in range(len(sizes) - 1, -1, -1):
        positions[i] = remaining % sizes[i]
        remaining //= sizes[i]

    return positions


# Prepare category codes and labels
category_codes = {}
category_labels = {}

for dimension in dimensions:
    category = dimension_metadata[dimension]["category"]

    category_codes[dimension] = ordered_categories(dimension)
    category_labels[dimension] = category.get("label", {})


# Convert sparse value/status objects into indexed maps
value_map = to_indexed_map(values)
status_map = to_indexed_map(status_values)


# Keep only observation indexes actually present
observation_indices = sorted(
    set(value_map.keys()) | set(status_map.keys())
)


records = []

for flat_index in observation_indices:

    positions = decode_flat_index(
        flat_index,
        sizes
    )

    record = {
        "source": "Eurostat",
        "dataset": "nasa_10_f_bs"
    }

    for dimension, position in zip(dimensions, positions):

        code = category_codes[dimension][position]

        record[dimension] = code

        labels = category_labels[dimension]

        if isinstance(labels, dict):
            record[f"{dimension}_label"] = labels.get(code)

    record["value"] = value_map.get(flat_index)
    record["status"] = status_map.get(flat_index)

    records.append(record)


df_pandas = pd.DataFrame(records)

# Ensure value is numeric
df_pandas["value"] = pd.to_numeric(
    df_pandas["value"],
    errors="coerce"
)

print("Observed rows:", len(df_pandas))
print("Columns:", len(df_pandas.columns))

display(df_pandas.head(20))

In [0]:
# ---------------------------------------------------------
# Initial data profiling
# ---------------------------------------------------------

print("Total observed rows:", len(df_pandas))
print("Null values:", df_pandas["value"].isna().sum())
print("Non-null values:", df_pandas["value"].notna().sum())

print("\nStatus distribution:")
print(
    df_pandas["status"]
    .value_counts(dropna=False)
)

print("\nUnique values by dimension:")

for column in [
    "freq",
    "unit",
    "co_nco",
    "sector",
    "finpos",
    "na_item",
    "geo",
    "time"
]:
    print(
        f"{column}: "
        f"{df_pandas[column].nunique()}"
    )

In [0]:
from pyspark.sql import functions as F

# ---------------------------------------------------------
# Create Spark DataFrame
# ---------------------------------------------------------

df_spark = spark.createDataFrame(df_pandas)

df_spark = (
    df_spark
    .withColumn(
        "ingestion_timestamp",
        F.current_timestamp()
    )
    .withColumn(
        "source_url",
        F.lit(
            "https://ec.europa.eu/eurostat/api/dissemination/"
            "statistics/1.0/data/nasa_10_f_bs"
        )
    )
)

print("Rows:", df_spark.count())
print("Columns:", len(df_spark.columns))

df_spark.printSchema()

In [0]:
# ---------------------------------------------------------
# Create Bronze Delta table
# ---------------------------------------------------------

table_name = "financial_balance_sheets_bronze"

(
    df_spark.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(table_name)
)

print(
    f"Bronze table created successfully: {table_name}"
)

In [0]:
# ---------------------------------------------------------
# Verify Bronze table
# ---------------------------------------------------------

bronze = spark.table(
    "financial_balance_sheets_bronze"
)

print("Bronze rows:", bronze.count())
print("Bronze columns:", len(bronze.columns))

display(bronze.limit(20))